# PipelineWatch-NG — Module 3: ML Anomaly Detection
**Goal:** Train and run ML models to classify:
1. **Oil spill blobs** (SAR dark spots → spill vs. look-alike using Random Forest)
2. **AIS vessel anomalies** (Isolation Forest on synthetic/real vessel movement data)
3. **Fire anomaly scoring** (XGBoost risk classifier on FIRMS clusters)

All models saved to Drive for use in Module 4.


## 3.1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/drive/MyDrive/PipelineWatch_NG'
PROCESSED    = f'{PROJECT_ROOT}/data/processed'
MODELS       = f'{PROJECT_ROOT}/models'
OUTPUTS      = f'{PROJECT_ROOT}/outputs'
os.makedirs(MODELS,  exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

!pip install xgboost scikit-learn joblib -q
print('✅ Environment ready.')


## 3.2 — Model A: SAR oil spill classifier (Random Forest)
Features extracted from each dark-spot blob to distinguish real spills from look-alikes (wind streaks, rain cells, upwelling).

In [ ]:
# Load SAR blob detections from Module 2
blobs_path = f'{PROCESSED}/sar_darkspot_blobs.csv'
sigma0_arr = np.load(f'{PROCESSED}/sigma0_vv.npy')

if os.path.exists(blobs_path):
    df_blobs = pd.read_csv(blobs_path)
    print(f'Loaded {len(df_blobs)} SAR blobs from Module 2')
else:
    df_blobs = pd.DataFrame()
    print('No blobs file found — will use synthetic.')

# Generate training dataset
# In production: label using NOSDRA spill reports + manual annotation
# Here: synthetic labels based on blob characteristics
np.random.seed(42)

def generate_sar_training_data(n=800):
    """
    Generate synthetic SAR blob feature dataset.
    Features known to distinguish oil spills from look-alikes:
      - mean_sigma0: spills are darker (-22 to -28 dB) than wind streaks (-18 to -21)
      - pixel_count: spills typically larger than rain cells
      - compactness: spills elongated (creeks), wind streaks very elongated
      - sigma0_std:  spills more homogeneous than wind roughness zones
      - aspect_ratio: width/height
    Labels: 1=oil spill, 0=look-alike
    """
    rows = []
    for _ in range(n):
        is_spill = np.random.binomial(1, 0.35)
        if is_spill:
            mean_s0  = np.random.normal(-24.5, 2.0)
            px_count = np.random.exponential(150) + 30
            s0_std   = np.random.uniform(0.5, 2.5)
            aspect   = np.random.uniform(1.5, 6.0)
            compact  = np.random.uniform(0.15, 0.55)
        else:  # look-alike
            mean_s0  = np.random.normal(-20.5, 2.5)
            px_count = np.random.exponential(60) + 10
            s0_std   = np.random.uniform(1.5, 5.0)
            aspect   = np.random.uniform(1.0, 12.0)
            compact  = np.random.uniform(0.05, 0.45)
        rows.append({
            'mean_sigma0': mean_s0,
            'pixel_count': min(px_count, 2000),
            'sigma0_std':  s0_std,
            'aspect_ratio':aspect,
            'compactness': compact,
            'label':       is_spill
        })
    return pd.DataFrame(rows)

df_train = generate_sar_training_data(1000)
print(f'SAR training set: {len(df_train)} blobs')
print(f'  Spill : {df_train["label"].sum()} ({df_train["label"].mean()*100:.0f}%)')
print(f'  Other : {(1-df_train["label"]).sum()}')


In [ ]:
# Feature matrix
FEATURES = ['mean_sigma0', 'pixel_count', 'sigma0_std', 'aspect_ratio', 'compactness']
X = df_train[FEATURES].values
y = df_train['label'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

y_pred = rf.predict(X_te)
print('=== SAR Spill Classifier — Test Set ===')
print(classification_report(y_te, y_pred, target_names=['look-alike','oil spill']))

# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='f1')
print(f'5-fold CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nFeature importances:')
print(fi.to_string())


In [ ]:
# Confusion matrix + feature importance plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Module 3 — SAR Oil Spill Classifier', fontsize=13)

ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred),
                       display_labels=['Look-alike','Oil spill']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion matrix')

fi.plot(kind='barh', ax=axes[1], color='#2196f3', edgecolor='white')
axes[1].set_title('Feature importance')
axes[1].set_xlabel('Importance')
plt.tight_layout()
fp = f'{OUTPUTS}/module3_sar_classifier.png'
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Figure saved → {fp}')

# Save model
joblib.dump(rf, f'{MODELS}/sar_spill_classifier.pkl')
joblib.dump(FEATURES, f'{MODELS}/sar_features.pkl')
print(f'✅ Model saved → {MODELS}/sar_spill_classifier.pkl')


## 3.3 — Model B: AIS vessel anomaly detection (Isolation Forest)
Detects vessels lingering in creek areas with unusual speed/heading patterns — signature of illegal bunkering.

In [ ]:
def generate_ais_data(n=2000):
    """
    Synthetic AIS vessel tracks for Niger Delta creeks.
    Normal vessels: transit at 8-14 knots, short dwell times
    Anomalous (bunkering): speed near 0, long dwell, inside creek
    """
    np.random.seed(99)
    rows = []
    for i in range(n):
        is_anomaly = np.random.binomial(1, 0.12)
        if is_anomaly:  # bunkering vessel
            lat     = np.random.uniform(4.2, 5.0)
            lon     = np.random.uniform(5.9, 6.8)
            speed   = np.random.exponential(0.8)           # nearly stationary
            heading = np.random.uniform(0, 360)
            dwell_h = np.random.exponential(8) + 4         # long stop
            dist_from_terminal_km = np.random.uniform(5, 40)
            dark_gap_h = np.random.exponential(6)          # AIS signal gaps
            vessel_type = np.random.choice([0,1,2], p=[0.05,0.85,0.10])  # mostly tanker
        else:           # legitimate transit
            lat     = np.random.uniform(4.2, 5.2)
            lon     = np.random.uniform(5.8, 7.2)
            speed   = np.random.normal(10, 2.5)
            heading = np.random.uniform(0, 360)
            dwell_h = np.random.exponential(0.5)
            dist_from_terminal_km = np.random.uniform(0.5, 100)
            dark_gap_h = np.random.exponential(0.5)
            vessel_type = np.random.choice([0,1,2], p=[0.5,0.3,0.2])
        rows.append({
            'mmsi': 100000000 + i,
            'lat': lat, 'lon': lon,
            'speed_kn':       max(0, speed),
            'heading':        heading,
            'dwell_hours':    max(0, dwell_h),
            'dist_terminal_km': dist_from_terminal_km,
            'ais_dark_gap_h': max(0, dark_gap_h),
            'vessel_type':    vessel_type,   # 0=cargo, 1=tanker, 2=other
            'true_anomaly':   is_anomaly
        })
    return pd.DataFrame(rows)

df_ais = generate_ais_data(2000)
print(f'AIS dataset: {len(df_ais)} vessel observations')
print(f'  True anomalies (bunkering): {df_ais["true_anomaly"].sum()}')
print(df_ais[['speed_kn','dwell_hours','dist_terminal_km','ais_dark_gap_h']].describe().round(2))


In [ ]:
AIS_FEATURES = ['speed_kn','dwell_hours','dist_terminal_km','ais_dark_gap_h','vessel_type']
X_ais = df_ais[AIS_FEATURES].values

scaler_ais = StandardScaler()
X_ais_sc   = scaler_ais.fit_transform(X_ais)

# Isolation Forest — unsupervised anomaly detection
iso = IsolationForest(contamination=0.12, n_estimators=200, random_state=42, n_jobs=-1)
iso.fit(X_ais_sc)

df_ais['anomaly_score'] = -iso.score_samples(X_ais_sc)   # higher = more anomalous
df_ais['is_anomaly']    = iso.predict(X_ais_sc) == -1     # -1 = anomaly

# Evaluate vs ground truth
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df_ais['true_anomaly'], df_ais['anomaly_score'])
print(f'Isolation Forest AUC-ROC: {auc:.3f}')
print()
print(f'Flagged as anomalous : {df_ais["is_anomaly"].sum()} vessels')
print(f'True positives       : {(df_ais["is_anomaly"] & df_ais["true_anomaly"]).sum()}')
print(f'False positives      : {(df_ais["is_anomaly"] & ~df_ais["true_anomaly"]).sum()}')
print()
print('Top 10 highest-risk vessels:')
top = df_ais.sort_values('anomaly_score', ascending=False).head(10)
print(top[['mmsi','lat','lon','speed_kn','dwell_hours','anomaly_score','true_anomaly']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Module 3 — AIS Vessel Anomaly Detection (Isolation Forest)', fontsize=13)

# Spatial scatter: anomaly score
anom = df_ais[df_ais['is_anomaly']]
norm = df_ais[~df_ais['is_anomaly']]
ax  = axes[0]
ax.scatter(norm['lon'], norm['lat'], s=8, alpha=0.3, c='#90a4ae', label='Normal')
ax.scatter(anom['lon'], anom['lat'], s=30, alpha=0.8, c='#e63946', label='Anomaly')
ax.set_xlim(5.7, 7.3); ax.set_ylim(4.1, 5.3)
ax.set_title('Vessel positions — anomalies flagged')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend()

# Score distribution
ax2 = axes[1]
ax2.hist(df_ais[~df_ais['true_anomaly']]['anomaly_score'], bins=40, alpha=0.6, label='Normal', color='#2196f3')
ax2.hist(df_ais[ df_ais['true_anomaly']]['anomaly_score'], bins=40, alpha=0.7, label='Bunkering', color='#e63946')
ax2.set_title('Anomaly score distribution')
ax2.set_xlabel('Anomaly score (higher = more suspicious)')
ax2.legend()
plt.tight_layout()
fp = f'{OUTPUTS}/module3_ais_anomaly.png'
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Figure saved → {fp}')

joblib.dump(iso,       f'{MODELS}/ais_isolation_forest.pkl')
joblib.dump(scaler_ais,f'{MODELS}/ais_scaler.pkl')
print(f'✅ AIS models saved.')


## 3.4 — Model C: Fire anomaly scorer (XGBoost)

In [ ]:
import xgboost as xgb

df_clusters = pd.read_csv(f'{PROCESSED}/firms_clusters.csv')

# Generate richer training set for XGBoost fire classifier
def generate_fire_training(n=600):
    np.random.seed(55)
    rows = []
    for _ in range(n):
        illegal = np.random.binomial(1, 0.30)
        if illegal:   # illegal refinery signature
            frp_sum  = np.random.exponential(400) + 100
            n_pix    = np.random.randint(10, 80)
            persist  = np.random.randint(3, 30)   # days active
            dist_road= np.random.exponential(8) + 2   # far from roads
            dist_pop = np.random.exponential(6) + 1   # far from settlements
            brightness = np.random.normal(360, 25)
        else:          # legitimate: gas flaring, agriculture
            frp_sum  = np.random.exponential(100) + 10
            n_pix    = np.random.randint(1, 20)
            persist  = np.random.randint(1, 8)
            dist_road= np.random.exponential(3)
            dist_pop = np.random.exponential(3)
            brightness = np.random.normal(330, 30)
        rows.append({
            'frp_sum_mw':    frp_sum,
            'n_pixels':      n_pix,
            'persistence_d': persist,
            'dist_road_km':  max(0, dist_road),
            'dist_pop_km':   max(0, dist_pop),
            'brightness':    brightness,
            'label':         illegal
        })
    return pd.DataFrame(rows)

df_fire_train = generate_fire_training(600)
FIRE_FEATURES = ['frp_sum_mw','n_pixels','persistence_d','dist_road_km','dist_pop_km','brightness']

Xf = df_fire_train[FIRE_FEATURES].values
yf = df_fire_train['label'].values
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, stratify=yf, random_state=42)

xgb_model = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08,
                                scale_pos_weight=2, use_label_encoder=False,
                                eval_metric='logloss', random_state=42)
xgb_model.fit(Xf_tr, yf_tr, eval_set=[(Xf_te, yf_te)], verbose=False)
yf_pred = xgb_model.predict(Xf_te)
print('=== Fire Anomaly Classifier (XGBoost) ===')
print(classification_report(yf_te, yf_pred, target_names=['Legitimate','Illegal refinery']))

joblib.dump(xgb_model,   f'{MODELS}/fire_xgb_classifier.pkl')
joblib.dump(FIRE_FEATURES,f'{MODELS}/fire_features.pkl')
print(f'\n✅ XGBoost fire model saved.')


## 3.5 — Module 3 summary

In [ ]:
print('='*52)
print('MODULE 3 — ML DETECTION COMPLETE')
print('='*52)
print()
print('Models trained and saved:')
for f in os.listdir(MODELS):
    sz = os.path.getsize(f'{MODELS}/{f}') / 1024
    print(f'  {f:<40} {sz:.1f} KB')
print()
print('Next → Module 4: Multi-source fusion & risk scoring')
